# Imports

In [12]:
import pandas as pd
import pint
import pint_pandas
ureg = pint.get_application_registry()

from aircraftdetective.processing.usdot import process_data_usdot_t2

from aircraftdetective.processing.literature import (
    process_data_weinold_database,
    process_data_babikian_figures
)

from aircraftdetective.calculations.engines import (
    determine_takeoff_to_cruise_tsfc_ratio,
    scale_engine_data_from_icao_emissions_database
)
from aircraftdetective.processing.acftdb import _read_engine_database
from aircraftdetective.utility.tabular import left_merge_wildcard

from aircraftdetective.calculations.engines import (
    calculate_air_mass_flow_rate,
    calculate_engine_efficiencies
)

from aircraftdetective.calculations.aerodynamics import (
    compute_lift_to_drag_ratio,
    compute_aspect_ratio
)

from aircraftdetective.utility.tabular import export_typed_dataframe_to_excel

In [4]:
df = process_data_weinold_database()
df_engines = _read_engine_database()

In [5]:
dict_tsfc_scaling = determine_takeoff_to_cruise_tsfc_ratio(degree=2, plot=True)
df_engines_scaled = scale_engine_data_from_icao_emissions_database(
    scaling_polynomial=dict_tsfc_scaling['TSFC (cruise)'],
)
df_engines_scaled.drop(columns=['Final Test Date'], inplace=True)

In [8]:
df_merged = left_merge_wildcard(
    df_left=df,
    df_right=df_engines_scaled,
    left_on='Engine Designation (ICAO)',
    right_on='Engine Identification',
)
df_merged = left_merge_wildcard(
    df_left=df_merged,
    df_right=df_engines,
    left_on='Engine Designation (aircraft-database.com)',
    right_on='Engine Designation',
)

In [10]:
df_merged = calculate_air_mass_flow_rate(df_merged)

In [11]:
df_t2 = process_data_usdot_t2()
df_with_dot = pd.merge(
    how='left',
    left=df_merged,
    right=df_t2,
    left_on='Aircraft Designation (US DOT Schedule T2)',
    right_on='Aircraft Designation (US DOT Schedule T2)'
)

In [13]:
df_with_dot = calculate_engine_efficiencies(df=df_with_dot)
df_with_dot = compute_aspect_ratio(df_with_dot)
df_with_dot = compute_lift_to_drag_ratio(df_with_dot, beta=0.04)

In [14]:
df_with_dot

,Manufacturer,Aircraft Designation,Aircraft Designation (Literature),Aircraft Designation (aircraft-database.com),Aircraft Designation (US DOT Schedule T2),Type,YOI,Engine Designation (ICAO),Engine Designation (aircraft-database.com),Design Range,...,Fuel/Available Seat Distance,Fuel/Revenue Seat Distance,Fuel Flow,Airborne Efficiency,SLF,Engine Efficiency,Propulsive Efficiency,Thermal Efficiency,Aspect Ratio,L/D
0,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,0.014860892639867753,0.017083725711339294,676.9628587507034,0.7947227191413238,0.8698859306786845,1.428333042570806e-06,0.8063583900473188,1.7713377329489786e-06,10.970703472840606,16.662689537570333
1,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,0.01436545664645943,0.016662296961420034,678.6354643254606,0.8202657470331677,0.8621534401722212,1.428333042570806e-06,0.8059727805181062,1.7721852115807508e-06,10.970703472840606,16.662689537570333
2,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,0.014551154123866285,0.017382073699310104,693.6825730359927,0.8202946900194606,0.8371356821737456,1.428333042570806e-06,0.8025202686738476,1.7798093061638236e-06,10.970703472840606,16.662689537570333
3,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,0.014568749605989218,0.017430667793303437,682.9326164514346,0.8241262216332615,0.835811328558868,1.428333042570806e-06,0.8049837858047291,1.774362499914113e-06,10.970703472840606,16.662689537570333
4,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,0.01418251936412575,0.021608873653475476,658.1251094063739,0.8329973838830038,0.6563284876185436,1.428333042570806e-06,0.8107269415694613,1.7617929901351742e-06,10.970703472840606,16.662689537570333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32916,McDonnell Douglas,DC-10-10,DC10-10,NaN,NaN,Wide,1971,NaN,NaN,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
32917,McDonnell Douglas,DC-10-30,DC10-30,NaN,NaN,Wide,1972,NaN,NaN,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
32918,McDonnell Douglas,DC-10-40,DC10-40,NaN,NaN,Wide,1973,NaN,NaN,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
32919,Lockheed,L-1011-100,L1011-1/100/200,NaN,NaN,Wide,1975,NaN,NaN,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
